# Earlier worked examples
Reproduce the original numerical examples and figures. Current manuscript commands and V3 data are listed in REPRODUCING.md.

In [ ]:
from pathlib import Path
import os, sys, json, subprocess
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'reproduce.py').exists())
sys.path.insert(0,str(ROOT/'src'))
sys.path.insert(0,str(ROOT/'scripts'))
for key in ['OPENBLAS_NUM_THREADS','OMP_NUM_THREADS','MKL_NUM_THREADS']: os.environ[key]='1'
import numpy as np
from qru_registerization.qru import random_qru_params
from qru_registerization.process import process_channel, process_metrics, diamond_choi
from check_paper import summarize
axis=[.3,-.4,np.sqrt(.75)]
xs=[-.7,.9]
print(json.dumps(summarize(),indent=2))

## Workspace policy
The same finite estimate is either retained coherently or renewed after every call. The echo is a score-independent diagnostic, not an improved score computation.

In [ ]:
p=random_qru_params(2,seed=600,scale=.6)
Z=[0,0,1]
X_target=np.kron(np.eye(2),np.array([[0,1],[1,0]]))
for protocol,gates in [('repetition',[np.eye(4)]),('echo',[X_target])]:
    for mode in ['shared','fresh']:
        result=process_channel(p,xs,axis,[(Z,.8)]*2,gates,phase_bits=3,mode=mode)
        values=process_metrics(result)
        print(protocol,mode,{k:values[k] for k in ['choi_trace_norm','diamond_upper']})

## Input mixing and the phase-sector boundary
Reconstruct the selected seed-603 example. The reference-assisted Choi witness already exceeds the extrapolated sector quantity, so the boundary is not solely an SDP observation.

In [ ]:
p=random_qru_params(2,seed=603,scale=.6)
Had=np.array([[1,1],[1,-1]])/np.sqrt(2)
R=np.kron(Had,np.array([[0,1],[1,0]]))
base=process_channel(p,xs,axis,[(Z,.8)]*4,[R]*3,phase_bits=3)
changed=process_channel(p,xs,axis,[(Z,.8)]*4,[R]*3,phase_bits=3,fibre=[0,2.1])
sector=2*np.linalg.eigvalsh(sum(k.conj().T@k for k in base.kraus[1::2]))[-1]
delta=changed.choi-base.choi
witness=np.linalg.norm(delta,ord='nuc')/4
print('2p1:',sector,'Choi witness:',witness,'Diamond witnesses:',diamond_choi(delta))
assert witness>sector+.05

## Certified renewal
Six score-dependent calls are grouped using clean-work channel certificates. Selection uses the certificate and structural-call budget.

In [ ]:
from qru_registerization.memory_contract import clean_contract, shared_bound
p=random_qru_params(2,seed=600,scale=.6)
result=process_channel(p,xs,axis,[(Z,.45)],phase_bits=3)
contract=clean_contract(result.zero_kraus,result.ideal)
print('One-call contract:',contract)
print('Six-call general guarantee:',shared_bound([contract]*6))
import csv
with (ROOT/'reference_data/renewal_study/policies.csv').open() as f:
    rows=list(csv.DictReader(f))
for row in rows:
    if row['seed']=='600' and row['phase_bits']=='3' and row['protocol']=='alternating':
        print({k:row[k] for k in ['policy','ends','a_calls','certificate','diamond_upper','readout_error']})

## Figures
Every numerical figure uses the archived campaign tables. Schematics are generated by vector drawing scripts, with labels and topology retaining their meaning in monochrome.

In [ ]:
subprocess.run([sys.executable,str(ROOT/'reproduce.py'),'figures','--version','v2'],cwd=ROOT,check=True)
from IPython.display import display, Image, Markdown
names=['00_research_overview','C01_qru_loading','C02_service_estimator','01_contract','02_precision','03_memory','04_fibre','05_noise','06_renewal','C03_grover_window','C04_target_control','A_resources','A_calibration','A_06_bit_allocation','A_03_scalar_vs_channel','A_07_quaternion_fibre','A_02_fibre_cancellation','A_04_dephasing_intervention','A_reuse','A01_depth','A02_bounds']
for name in names:
    display(Markdown('### '+name.replace('_',' ')))
    display(Image(filename=str(ROOT/'outputs/figures'/f'{name}.png'),width=700))

## Complete numerical rerun
Run the following separately to regenerate all nine campaigns, compare their CSVs to the reference tables and render figures from the recomputed data. The scripts write progress under `outputs/results/`.

In [ ]:
# subprocess.run([sys.executable,'reproduce.py','experiments','--version','v2'],cwd=ROOT,check=True)
# subprocess.run([sys.executable,'reproduce.py','compare'],cwd=ROOT,check=True)
# subprocess.run([sys.executable,'reproduce.py','figures','--version','v2','--data','outputs/results'],cwd=ROOT,check=True)